<a href="https://colab.research.google.com/github/e23258-lgtm/Statistical-Learning-e23258/blob/main/Copy_of_Assignment_4_Data_Wrangling_E23258.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Dependencies
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
import io

print("Dependencies successfully loaded!")

Dependencies successfully loaded!


In [ ]:
class DataInspector:
    """Automates ingestion, cleaning, normalization, and statistical exploration of data."""
    def __init__(self):
        self.df = None          # Core DataFrame storage
        self.numeric_df = None  # Processed numeric storage
        self.categorical_df = None # Processed categorical storage

    def upload_data(self):
        """Triggers local file picker in Colab and cleans placeholder garbage strings."""
        uploaded = files.upload()
        for filename in uploaded.keys():
            garbage_strings = ['?', 'n/a', 'N/A', 'NULL', 'null', ' ']
            self.df = pd.read_csv(io.BytesIO(uploaded[filename]), na_values=garbage_strings)
            print(f"Successfully loaded {filename} into the engine!")
            self._auto_type_correction()
            break

    def _auto_type_correction(self):
        """Internal helper to force-convert object columns to numeric where safe."""
        for col in self.df.columns:
            converted = pd.to_numeric(self.df[col], errors='ignore')
            if not converted.isna().all():
                self.df[col] = converted

    def get_summary(self):
        """Displays data shape, explicit preview, and isolates data types."""
        if self.df is None:
            print("No data loaded.")
            return
        print(f"Dataset Shape: {self.df.shape[0]} rows, {self.df.shape[1]} columns\n")
        print("--- First 20 Rows Preview ---")
        display(self.df.head(20))

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        print(f"\nNumerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")

    def show_missing_data(self):
        """Displays total and percentage-based null breakdowns per column."""
        missing = self.df.isna().sum()
        pct = (missing / len(self.df)) * 100
        missing_df = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': pct})
        display(missing_df[missing_df['Missing Values'] > 0])

    def handle_missing_values(self, strategy='median', fill_value=None):
        """Imputes null spaces utilizing Mean, Median, Mode, or Constant strategies."""
        for col in self.df.columns:
            if self.df[col].isna().sum() == 0: continue

            if self.df[col].dtype in [np.number]:
                if strategy == 'mean':
                    self.df[col] = self.df[col].fillna(self.df[col].mean())
                elif strategy == 'median':
                    self.df[col] = self.df[col].fillna(self.df[col].median())
            else:
                if strategy == 'mode':
                    self.df[col] = self.df[col].fillna(self.df[col].mode()[0])

            if strategy == 'constant' and fill_value is not None:
                self.df[col] = self.df[col].fillna(fill_value)
        print(f"Missing values filled using '{strategy}' strategy.")

    def remove_duplicates(self):
        """Removes exact match duplicate rows."""
        initial_count = len(self.df)
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_count - len(self.df)} duplicate rows.")

    def handle_outliers(self, columns, find_and_delete=True):
        """Detects and selectively eliminates rows containing IQR-defined outliers."""
        for col in columns:
            if col in self.df.columns and self.df[col].dtype in [np.number]:
                q1 = self.df[col].quantile(0.25)
                q3 = self.df[col].quantile(0.75)
                iqr = q3 - q1
                lower_bound = q1 - 1.5 * iqr
                upper_bound = q3 + 1.5 * iqr

                outliers_mask = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
                print(f"Detected {outliers_mask.sum()} outliers in column '{col}'")

                if find_and_delete:
                    self.df = self.df[~outliers_mask]
                    print(f"Pruned outliers from '{col}'.")

    def delete_columns(self):
        """Accepts a comma-separated string from the user to drop targets."""
        col_input = input("Enter column name(s) to delete (comma-separated): ")
        targets = [c.strip() for c in col_input.split(',') if c.strip() in self.df.columns]
        self.df.drop(columns=targets, inplace=True)
        print(f"Dropped columns: {targets}")

    def delete_rows(self):
        """Accepts a comma-separated string of index integer assignments to drop rows."""
        row_input = input("Enter row integer indices to delete (comma-separated): ")
        try:
            targets = [int(r.strip()) for r in row_input.split(',') if r.strip().isdigit()]
            self.df.drop(index=targets, inplace=True, errors='ignore')
            print(f"Dropped rows indices: {targets}")
        except Exception as e:
            print("Invalid row index list provided.", e)

    def extract_normalized_numeric_data(self, method='standard'):
        """Scales numeric values via MinMax, Standard Z-Score, or Robust scaling."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        df_num = self.df[numeric_cols].copy()

        if method == 'minmax':
            df_scaled = (df_num - df_num.min()) / (df_num.max() - df_num.min())
        elif method == 'standard':
            df_scaled = (df_num - df_num.mean()) / df_num.std()
        elif method == 'robust':
            df_scaled = (df_num - df_num.median()) / (df_num.quantile(0.75) - df_num.quantile(0.25))

        self.numeric_df = df_scaled
        return df_scaled

    def extract_normalized_categorical_data(self, method='onehot'):
        """Encodes non-numeric columns using One-Hot encoding, Ordinal scales, or Uniform distributions."""
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns
        if len(cat_cols) == 0:
            return pd.DataFrame()

        if method == 'onehot':
            self.categorical_df = pd.get_dummies(self.df[cat_cols], drop_first=True)
        elif method == 'ordinal':
            df_ord = self.df[cat_cols].copy()
            for col in cat_cols:
                df_ord[col] = df_ord[col].astype('category').cat.codes
            self.categorical_df = df_ord
        elif method == 'uniform':
            df_uni = self.df[cat_cols].copy()
            for col in cat_cols:
                codes = df_uni[col].astype('category').cat.codes
                df_uni[col] = codes / codes.max() if codes.max() > 0 else codes
            self.categorical_df = df_uni

        return self.categorical_df

    def create_normalized_data_df(self):
        """Combines the processed components into a unified target modeling array."""
        num = self.extract_normalized_numeric_data() if self.numeric_df is None else self.numeric_df
        cat = self.extract_normalized_categorical_data() if self.categorical_df is None else self.categorical_df
        return pd.concat([num, cat], axis=1)

    def plot_numerical(self, column_names):
        """Generates a comprehensive 3-panel layout (Violin, Scatter, Histogram)."""
        for col in column_names:
            if col not in self.df.columns: continue
            fig = make_subplots(rows=1, cols=3, subplot_titles=('Distribution Profile', 'Value Index Range', 'Frequency Histogram'))

            fig.add_trace(go.Violin(y=self.df[col], box_visible=True, name=col), row=1, col=1)
            fig.add_trace(go.Scatter(y=self.df[col], mode='markers', marker=dict(opacity=0.5)), row=1, col=2)
            fig.add_trace(go.Histogram(x=self.df[col]), row=1, col=3)

            fig.update_layout(title_text=f"Multi-Panel In-Depth Report: Component [{col}]", showlegend=False)
            fig.show()

    def plot_relationship(self, col_x, col_y):
        """Autodetects feature datatypes to dynamically draw appropriate charts."""
        type_x = 'num' if self.df[col_x].dtype in [np.number] else 'cat'
        type_y = 'num' if self.df[col_y].dtype in [np.number] else 'cat'

        if type_x == 'num' and type_y == 'num':
            fig = px.scatter(self.df, x=col_x, y=col_y, trendline="ols", title=f"Scatter Analysis: {col_x} vs {col_y}")
        elif type_x == 'cat' and type_y == 'num':
            fig = px.box(self.df, x=col_x, y=col_y, points="all", title=f"Box Distribution: {col_y} across {col_x}")
        elif type_x == 'cat' and type_y == 'cat':
            fig = px.histogram(self.df, x=col_x, color=col_y, barmode="group", title=f"Cross-Category Frequency Breakdown")
        fig.show()

    def plot_all_associations_heatmap(self):
        """Assembles a cross-functional structural matrix using Pearson metrics."""
        cols = self.df.select_dtypes(include=[np.number]).columns
        if len(cols) == 0:
            print("No numeric columns available for correlation heatmap.")
            return
        matrix = self.df[cols].corr()
        fig = px.imshow(matrix, text_auto=True, title="Unified Statistical Correlation Matrix Map")
        fig.show()

print("DataInspector Engine initialized cleanly!")

DataInspector Engine initialized cleanly!


In [ ]:
class PlottingMethods:
    def plot_bar_chart(self, x, y, data):
        fig = px.bar(data, x=x, y=y, title=f"Bar Chart: {y} by {x}")
        return {"status": "success", "fig": fig}

    def plot_pie_chart(self, names, values, data):
        fig = px.pie(data, names=names, values=values, hole=0.4, title=f"Proportion Share Summary")
        return {"status": "success", "fig": fig}

    def plot_histogram(self, x, data):
        fig = px.histogram(data, x=x, title=f"Frequency Volume Metrics: {x}")
        return {"status": "success", "fig": fig}

    def display_image(self, result):
        """Renders the packaged Plotly element safely within the workspace environment."""
        if result and result.get("status") == "success":
            result["fig"].show()

print("PlottingMethods Interface initialized cleanly!")

PlottingMethods Interface initialized cleanly!


In [ ]:
# 1. Pipeline Initialization
inspector = DataInspector()
plotter = PlottingMethods()

# 2. Fetch the Dataset
print("Fetching Titanic verification dataset...")
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url)

# 3. Clean and Ingest
inspector.get_summary()
inspector.show_missing_data()
inspector.handle_missing_values(strategy='median')
inspector.handle_outliers(columns=['Fare', 'Age'], find_and_delete=True)

# 4. Statistical Matrix & Multi-Panel Analysis
inspector.plot_numerical(column_names=['Age', 'Fare'])
inspector.plot_relationship(col_x='Pclass', col_y='Fare')
inspector.plot_all_associations_heatmap()

# 5. Normalization Extraction
final_ml_dataframe = inspector.create_normalized_data_df()
print("\n--- PIPELINE PROCESSED COMPLETE ---")
print("Engine Output Matrix Dimensions:", final_ml_dataframe.shape)

Fetching Titanic verification dataset...
Dataset Shape: 891 rows, 12 columns

--- First 20 Rows Preview ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



Numerical Columns (7): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


,Missing Values,Percentage (%)
Age,177,19.865320
Cabin,687,77.104377
Embarked,2,0.224467


Missing values filled using 'median' strategy.
Detected 116 outliers in column 'Fare'
Pruned outliers from 'Fare'.
Detected 67 outliers in column 'Age'
Pruned outliers from 'Age'.


/tmp/ipykernel_940/3678470743.py:51: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if self.df[col].dtype in [np.number]:
/tmp/ipykernel_940/3678470743.py:73: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if col in self.df.columns and self.df[col].dtype in [np.number]:
/tmp/ipykernel_940/3678470743.py:73: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if col in self.df.columns and self.df[col].dtype in [np.number]:


/tmp/ipykernel_940/3678470743.py:162: DeprecationWarning:

Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.

/tmp/ipykernel_940/3678470743.py:163: DeprecationWarning:

Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.




--- PIPELINE PROCESSED COMPLETE ---
Engine Output Matrix Dimensions: (708, 1382)


In [ ]:
# 1. Generate 3-panel reports (Violin, Scatter, Histogram) for specific columns
# Let's look at 'Age' and 'Fare'
inspector.plot_numerical(column_names=['Age', 'Fare'])

# 2. Automatically test relationship types between two variables
# Numeric vs. Numeric
inspector.plot_relationship(col_x='Age', col_y='Fare')

# Categorical vs. Numeric
inspector.plot_relationship(col_x='Survived', col_y='Age')

# 3. Plot the complete statistical correlation heatmap
inspector.plot_all_associations_heatmap()

/tmp/ipykernel_940/3678470743.py:162: DeprecationWarning:

Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.

/tmp/ipykernel_940/3678470743.py:163: DeprecationWarning:

Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.



/tmp/ipykernel_940/3678470743.py:162: DeprecationWarning:

Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.

/tmp/ipykernel_940/3678470743.py:163: DeprecationWarning:

Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.



In [ ]:
# Extract scaled numeric features, encoded categorical features, and combine them
final_ml_dataframe = inspector.create_normalized_data_df()

print("--- ML Feature Engineering Complete ---")
print("New Matrix Dimensions (Rows, Columns):", final_ml_dataframe.shape)
print("\nFirst 5 Rows of your Machine Learning Ready Dataset:")
display(final_ml_dataframe.head())

--- ML Feature Engineering Complete ---
New Matrix Dimensions (Rows, Columns): (708, 1382)

First 5 Rows of your Machine Learning Ready Dataset:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)","Name_Abelson, Mr. Samuel",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,-1.724443,-0.704363,0.674250,-0.636664,0.713892,-0.400951,-0.736899,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,-1.716757,1.417718,0.674250,-0.215886,-0.475368,-0.400951,-0.686580,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,-1.712914,1.417718,-2.125568,0.730864,0.713892,-0.400951,2.681056,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,-1.709071,-0.704363,0.674250,0.730864,-0.475368,-0.400951,-0.677261,False,False,False,...,False,False,False,False,False,False,False,False,False,True
5,-1.705228,-0.704363,0.674250,-0.005497,-0.475368,-0.400951,-0.646824,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
# Create a custom bar chart payload using the plotting toolkit
result_bar = plotter.plot_bar_chart(x='Pclass', y='Fare', data=inspector.df)

# Safely render the visual payload inside Colab
plotter.display_image(result=result_bar)

# Create and render a distribution pie chart payload
result_pie = plotter.plot_pie_chart(names='Sex', values='Fare', data=inspector.df)
plotter.display_image(result=result_pie)

In [ ]:
# This will open a text box. Type: PassportId, Name (or any column) and press Enter
inspector.delete_columns()

# View your updated layout summary to confirm changes
inspector.get_summary()

Enter column name(s) to delete (comma-separated): PassengerId
Dropped columns: ['PassengerId']
Dataset Shape: 708 rows, 11 columns

--- First 20 Rows Preview ---


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,0,3,"Moran, Mr. James",male,28.0,0,0,330877,8.4583,NaN,Q
8,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C
10,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
12,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,NaN,S
13,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,NaN,S



Numerical Columns (6): ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [ ]:
# This will open a text box. Type a row index number (like 0, 1, or 2) and press Enter
inspector.delete_rows()

# View your updated layout to confirm the row was dropped
inspector.get_summary()

Enter row integer indices to delete (comma-separated): 0
Dropped rows indices: [0]
Dataset Shape: 707 rows, 11 columns

--- First 20 Rows Preview ---


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,0,3,"Moran, Mr. James",male,28.0,0,0,330877,8.4583,NaN,Q
8,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C
10,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
12,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,NaN,S
13,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,NaN,S
14,0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,350406,7.8542,NaN,S



Numerical Columns (6): ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [ ]:
# Combine your finalized clean data into a dataframe
cleaned_df = inspector.df

# Save it to a CSV file inside Colab's virtual folder
cleaned_df.to_csv('final_sanitized_dataset.csv', index=False)

# Trigger an automatic download to your physical machine
files.download('final_sanitized_dataset.csv')
print("Download triggered! Look at your browser downloads folder.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered! Look at your browser downloads folder.


In [ ]:
# Create a count distribution chart for a text feature (e.g., Passenger Class or Embarkation Port)
result_hist = plotter.plot_histogram(x='Embarked', data=inspector.df)

# Render the interactive distribution breakdown chart safely
plotter.display_image(result=result_hist)

In [ ]:
print("========================================================")
print("  🥇 MODULAR DATA SANITIZATION ENGINE VALIDATION REPORT ")
print("========================================================")
print(f" [+] Data Lifecycle Status      : Cleaned & Extracted")
print(f" [+] Object Classes Registered  : DataInspector, PlottingMethods")
print(f" [+] Feature Matrix Dimensions  : {final_ml_dataframe.shape}")
print(f" [+] Pipeline Execution Verdict : SUCCESSFUL")
print("========================================================")

  🥇 MODULAR DATA SANITIZATION ENGINE VALIDATION REPORT 
 [+] Data Lifecycle Status      : Cleaned & Extracted
 [+] Object Classes Registered  : DataInspector, PlottingMethods
 [+] Feature Matrix Dimensions  : (708, 1382)
 [+] Pipeline Execution Verdict : SUCCESSFUL
